## The Supply Chain Autopsy

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd

In [ ]:
# load perviously processed data ("delivery_delay_hypothesis")
date_cols = [
    "order_purchase_timestamp",
     "order_approved_at",
     "order_delivered_carrier_date",
     "order_delivered_customer_date",
     "order_estimated_delivery_date",
     "review_creation_date",
     "review_answer_timestamp"
]
delivery_delay_hypothesis = pd.read_csv("../data/processed/delivery_delay_hypothesis.csv", parse_dates=date_cols)

In [ ]:
# Load order_items dataset
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
order_items.info()

In [ ]:
# Load sellers dataset
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
sellers.shape

In [ ]:
# Merge delivery_delay_hypothesis and order_items dataset.
df = pd.merge(delivery_delay_hypothesis, order_items, on="order_id", how="left")
df = pd.merge(df, sellers, on="seller_id", how="left")
df.sample(3)

In [ ]:
# Drop the NaT in required columns
df = df.dropna(subset=["order_approved_at", "order_delivered_carrier_date"])

In [ ]:
df["seller_processing_days"] = (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.days

In [ ]:
# Insights
seller_metrics = df.groupby("seller_id").agg(
    total_volume = ("order_item_id", "count"),
    total_orders = ("order_id", "nunique"),
    avg_processing_time = ("seller_processing_days", "mean"),
    avg_delay = ("delivery_delay_days", "mean"),
    total_delayed_orders = ('delivery_delay_days', lambda x: (x > 0).sum())
).reset_index()

seller_metrics["avg_processing_time"] = seller_metrics["avg_processing_time"].round(2) 
seller_metrics["avg_delay"] = seller_metrics["avg_delay"].round(2) 

In [ ]:
min_orders = 50
on_time_sellers = seller_metrics[seller_metrics['total_orders'] >= min_orders].copy()

In [ ]:
print(f"Original sellers: {len(seller_metrics)}")
print(f"Sellers with {min_orders}+ orders: {len(on_time_sellers)}")
print(f"Removed: {len(seller_metrics) - len(on_time_sellers)} low-volume sellers")

##### Top 10 sellers

In [ ]:
top_10_sellers = on_time_sellers.sort_values(by="avg_delay", ascending=False).reset_index().head(10)

In [ ]:
top_10_sellers

In [ ]:
# Export processed data
# df.to_csv("../data/processed/supply_chain_autopsy.csv", index=False)

### Observation

Among high-volume sellers (50+ orders), average processing time (seller_processing_days) does not strongly correlate with average delay, suggesting that delays occur more frequently during shipping and logistics rather than at the seller's preparation stage.